# Aula introdutória de Pandas: lendo Excel e tratando dados

Objetivo desta aula: sair sabendo **ler um Excel real, explorar os dados, limpar problemas comuns e extrair valores** para usar em outros lugares (relatórios, cálculos, decisões).

Pré-requisito: Python básico (variáveis, listas, dicionários, laços, funções). Não é necessário saber pandas.

**Estrutura:**
1. O que é o pandas (Series x DataFrame)
2. Lendo um arquivo Excel
3. Explorando os dados (primeiro olhar)
4. Selecionando linhas e colunas
5. Tratando dados sujos (nulos, duplicados, tipos, texto inconsistente)
6. Criando colunas novas e transformando dados
7. Agrupando e resumindo (groupby)
8. Ordenando e filtrando
9. Salvando o resultado
10. Exercícios propostos


## 0. Preparando o ambiente

Vamos usar a biblioteca `pandas` (manipulação de dados) e `openpyxl` (motor que o pandas usa por baixo dos panos para ler `.xlsx`).

Se não tiver instalado, rode no terminal:
```
pip install pandas openpyxl
```


In [2]:
import pandas as pd
import numpy as np

# Configuração só para exibir mais colunas/linhas no notebook, sem cortar com "..."
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Versão do pandas:", pd.__version__)

import os
print(os.getcwd())


Versão do pandas: 3.0.5
/home/polo/Documents/superProf/aulaPandas


## 1. Series x DataFrame

Pandas tem duas estruturas principais:

- **Series**: uma coluna, tipo uma lista com "rótulos" (índice).
- **DataFrame**: uma tabela inteira, com linhas e colunas — pense numa planilha do Excel dentro do Python.

Todo Excel que você lê vira um **DataFrame**.


In [18]:
# Exemplo simples só para fixar o conceito, sem depender de arquivo nenhum
serie = pd.Series([10, 20, 30], name="idade")
print(serie)
print("------")
print(type(serie))


0    10
1    20
2    30
Name: idade, dtype: int64
------
<class 'pandas.Series'>


In [20]:
df_exemplo = pd.DataFrame({
    "nome": ["Ana", "Bruno", "Carla"],
    "idade": [23, 35, 29],
    "peso": [55, 98, 75]
})
df_exemplo  # em notebook, retornar o DataFrame na última linha já exibe como tabela


,nome,idade,peso
0,Ana,23,55
1,Bruno,35,98
2,Carla,29,75


## 2. Lendo um arquivo Excel

Vamos usar um arquivo de exemplo (`vendas_exemplo.xlsx`) que simula um relatório de vendas real — **de propósito com problemas comuns**: um valor nulo, uma linha duplicada, e inconsistência de maiúscula/minúscula. Isso é para você já treinar a limpeza também.

A função principal é `pd.read_excel()`.


In [24]:
caminho = "~/Documents/superProf/aulaPandas/vendas_exemplo.xlsx"  # ajuste o caminho se o arquivo estiver em outra pasta

df = pd.read_excel(caminho, sheet_name="Vendas")  # sheet_name é opcional se só tiver uma aba
df



,ID_Venda,Cliente,Produto,Categoria,Quantidade,Preco_Unitario,Data_Venda,Estado
0,1,Ana Souza,Notebook,Eletrônicos,1.0,3500.0,2025-01-05,SP
1,2,Bruno Lima,Mouse,Acessórios,2.0,89.9,2025-01-06,sp
2,3,Carla Dias,Teclado,Acessórios,1.0,150.5,2025-01-06,RJ
3,4,Bruno Lima,Mouse,Acessórios,3.0,89.9,2025-01-07,SP
4,5,Eduardo Reis,Monitor,Eletrônicos,1.0,1200.0,2025-01-08,MG
5,6,Fernanda Melo,Notebook,Eletrônicos,1.0,3500.0,2025-01-09,SP
6,7,NaN,Cadeira,Móveis,1.0,650.0,2025-01-10,RJ
7,8,Helena Costa,Monitor,Eletrônicos,2.0,1200.0,2025-01-10,MG
8,9,Igor Alves,Mouse,Acessórios,NaN,89.9,2025-01-11,SP
9,10,Julia Prado,Teclado,Acessórios,1.0,150.5,2025-01-12,RJ


**Parâmetros úteis do `read_excel` que vale conhecer:**

| Parâmetro | Para que serve |
|---|---|
| `sheet_name` | qual aba ler (nome ou número, ou `None` para ler todas como dicionário) |
| `header` | qual linha é o cabeçalho (padrão é a linha 0) |
| `usecols` | quais colunas ler, ex: `"A:C"` ou `["Cliente", "Produto"]` |
| `skiprows` | pular linhas do topo (útil quando a planilha tem título antes da tabela) |
| `dtype` | forçar o tipo de uma coluna na leitura |
| `na_values` | valores extras que devem virar nulo, ex: `["N/A", "-"]` |


## 3. Primeiro olhar nos dados

Antes de tratar qualquer coisa, é essencial **entender o que você recebeu**. Esses comandos são os primeiros que eu sempre rodo em qualquer planilha nova.


In [6]:
df.shape  # (número de linhas, número de colunas)


(12, 8)

In [26]:
df.head()  # primeiras 5 linhas (pode passar um número, ex: df.head(10))


,ID_Venda,Cliente,Produto,Categoria,Quantidade,Preco_Unitario,Data_Venda,Estado
0,1,Ana Souza,Notebook,Eletrônicos,1.0,3500.0,2025-01-05,SP
1,2,Bruno Lima,Mouse,Acessórios,2.0,89.9,2025-01-06,sp
2,3,Carla Dias,Teclado,Acessórios,1.0,150.5,2025-01-06,RJ
3,4,Bruno Lima,Mouse,Acessórios,3.0,89.9,2025-01-07,SP
4,5,Eduardo Reis,Monitor,Eletrônicos,1.0,1200.0,2025-01-08,MG


In [29]:
df.info()  # tipo de cada coluna + quantos valores não-nulos existem -- ótimo para achar nulos rápido


<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID_Venda        12 non-null     int64  
 1   Cliente         11 non-null     str    
 2   Produto         12 non-null     str    
 3   Categoria       12 non-null     str    
 4   Quantidade      11 non-null     float64
 5   Preco_Unitario  12 non-null     float64
 6   Data_Venda      12 non-null     str    
 7   Estado          12 non-null     str    
dtypes: float64(2), int64(1), str(5)
memory usage: 900.0 bytes


In [9]:
df.describe()  # estatísticas básicas das colunas numéricas (média, min, max, etc.)


,ID_Venda,Quantidade,Preco_Unitario
count,12.000000,11.000000,12.000000
mean,6.416667,1.363636,1189.266667
std,3.502164,0.674200,1451.407916
min,1.000000,1.000000,89.900000
25%,3.750000,1.000000,135.350000
50%,6.500000,1.000000,400.250000
75%,9.250000,1.500000,1775.000000
max,12.000000,3.000000,3500.000000


In [10]:
df.dtypes  # tipo de dado de cada coluna, isolado


ID_Venda            int64
Cliente               str
Produto               str
Categoria             str
Quantidade        float64
Preco_Unitario    float64
Data_Venda            str
Estado                str
dtype: object

Olhando o resultado acima, já dá pra notar alguns problemas:
- `Cliente` tem menos valores não-nulos que o total de linhas → **tem nulo**
- `Quantidade` também tem nulo
- `Data_Venda` provavelmente está como texto (`object`), não como data


## 4. Selecionando linhas e colunas

Formas principais de selecionar dados:


In [31]:
# Uma coluna -> vira uma Series
df["Cliente"]


0         Ana Souza
1        Bruno Lima
2        Carla Dias
3        Bruno Lima
4      Eduardo Reis
5     Fernanda Melo
6               NaN
7      Helena Costa
8        Igor Alves
9       Julia Prado
10      Julia Prado
11      Karen Nunes
Name: Cliente, dtype: str

In [12]:
# Várias colunas -> passa uma lista, vira um DataFrame
df[["Cliente", "Produto", "Preco_Unitario"]]


,Cliente,Produto,Preco_Unitario
0,Ana Souza,Notebook,3500.0
1,Bruno Lima,Mouse,89.9
2,Carla Dias,Teclado,150.5
3,Bruno Lima,Mouse,89.9
4,Eduardo Reis,Monitor,1200.0
5,Fernanda Melo,Notebook,3500.0
6,NaN,Cadeira,650.0
7,Helena Costa,Monitor,1200.0
8,Igor Alves,Mouse,89.9
9,Julia Prado,Teclado,150.5


In [32]:
# loc: seleciona por RÓTULO (nome da linha/coluna)
df.loc[4:9, ["Cliente", "Produto"]]  # linhas de índice 0 a 4, só essas colunas


,Cliente,Produto
4,Eduardo Reis,Monitor
5,Fernanda Melo,Notebook
6,NaN,Cadeira
7,Helena Costa,Monitor
8,Igor Alves,Mouse
9,Julia Prado,Teclado


In [14]:
# iloc: seleciona por POSIÇÃO (número), como se fosse uma lista/matriz
df.iloc[0:3, 0:2]  # 3 primeiras linhas, 2 primeiras colunas


,ID_Venda,Cliente
0,1,Ana Souza
1,2,Bruno Lima
2,3,Carla Dias


In [39]:
# Filtro booleano: a forma mais usada no dia a dia
df[df["Produto"] == "Notebook"]


,ID_Venda,Cliente,Produto,Categoria,Quantidade,Preco_Unitario,Data_Venda,Estado
0,1,Ana Souza,Notebook,Eletrônicos,1.0,3500.0,2025-01-05,SP
5,6,Fernanda Melo,Notebook,Eletrônicos,1.0,3500.0,2025-01-09,SP
11,12,Karen Nunes,Notebook,Eletrônicos,1.0,3500.0,2025-01-13,SP


In [37]:
# Combinando condições: & é "e", | é "ou" -- SEMPRE use parênteses em cada condição
df[(df["Produto"] == "Monitor") & (df["Estado"] == "MG")]


,ID_Venda,Cliente,Produto,Categoria,Quantidade,Preco_Unitario,Data_Venda,Estado
4,5,Eduardo Reis,Monitor,Eletrônicos,1.0,1200.0,2025-01-08,MG
7,8,Helena Costa,Monitor,Eletrônicos,2.0,1200.0,2025-01-10,MG


## 5. Tratando dados sujos

Toda planilha real do mundo real vem com problemas. Vamos tratar um por um.


### 5.1 Valores nulos (`NaN`)

In [ ]:
# Quantos nulos existem por coluna
df.isna().sum()


In [ ]:
# Ver só as linhas que têm algum nulo
df[df.isna().any(axis=1)]


In [ ]:
# Opção A: remover linhas com nulo (cuidado: você perde a linha inteira)
df_sem_nulo = df.dropna()
df_sem_nulo.shape


In [ ]:
# Opção B: preencher o nulo com um valor (mais comum na prática)
df_tratado = df.copy()  # sempre trabalhe numa cópia para não perder o original

df_tratado["Cliente"] = df_tratado["Cliente"].fillna("Cliente não identificado")
df_tratado["Quantidade"] = df_tratado["Quantidade"].fillna(1)  # aqui decidimos assumir 1 unidade

df_tratado.isna().sum()


### 5.2 Linhas duplicadas

In [ ]:
# Ver quais linhas estão duplicadas (considerando todas as colunas)
df_tratado[df_tratado.duplicated()]


In [ ]:
# Remover duplicadas, mantendo a primeira ocorrência
df_tratado = df_tratado.drop_duplicates()
df_tratado.shape


In [ ]:
# Às vezes a duplicidade é por uma coluna específica, ex: mesmo ID_Venda
df_tratado[df_tratado.duplicated(subset="ID_Venda")]


### 5.3 Inconsistência de texto (maiúscula/minúscula, espaços)

In [ ]:
# Repare que 'SP' e 'sp' são tratados como valores diferentes -- problema clássico
df_tratado["Estado"].unique()


In [ ]:
# Padronizar para maiúsculo e remover espaços extras nas pontas
df_tratado["Estado"] = df_tratado["Estado"].str.strip().str.upper()
df_tratado["Estado"].unique()


### 5.4 Corrigindo tipos de dado (`dtype`)

In [ ]:
# Data_Venda veio como texto -- vamos converter para datetime de verdade
df_tratado["Data_Venda"] = pd.to_datetime(df_tratado["Data_Venda"])
df_tratado.dtypes


In [ ]:
# Agora dá pra usar propriedades de data, por exemplo extrair o dia da semana
df_tratado["Dia_Semana"] = df_tratado["Data_Venda"].dt.day_name()
df_tratado[["Data_Venda", "Dia_Semana"]]


**Resumo das ferramentas de limpeza:**

| Problema | Ferramenta |
|---|---|
| Valor nulo | `isna()`, `dropna()`, `fillna()` |
| Linha duplicada | `duplicated()`, `drop_duplicates()` |
| Texto inconsistente | `.str.strip()`, `.str.upper()`, `.str.lower()`, `.str.replace()` |
| Tipo errado | `astype()`, `pd.to_datetime()`, `pd.to_numeric()` |


## 6. Criando colunas novas e usando os valores

Esse é o passo onde você efetivamente **usa os valores** da planilha para calcular algo novo.


In [ ]:
# Coluna calculada a partir de outras duas -- é só fazer conta com as colunas direto
df_tratado["Valor_Total"] = df_tratado["Quantidade"] * df_tratado["Preco_Unitario"]
df_tratado[["Produto", "Quantidade", "Preco_Unitario", "Valor_Total"]]


In [ ]:
# Acessando um valor específico (ex: o valor total da primeira venda)
valor = df_tratado.loc[0, "Valor_Total"]
print("Valor total da venda 0:", valor)


In [ ]:
# apply(): quando a lógica é mais complexa que uma conta simples
def classificar_venda(valor):
    if valor >= 1000:
        return "Alto"
    elif valor >= 200:
        return "Médio"
    else:
        return "Baixo"

df_tratado["Porte_Venda"] = df_tratado["Valor_Total"].apply(classificar_venda)
df_tratado[["Produto", "Valor_Total", "Porte_Venda"]]


## 7. Agrupando e resumindo (`groupby`)

`groupby` é uma das ferramentas mais usadas do pandas: agrupa linhas por uma coluna e aplica um cálculo (soma, média, contagem...) em cada grupo. É basicamente uma **tabela dinâmica** feita em código.


In [ ]:
# Total vendido por Estado
df_tratado.groupby("Estado")["Valor_Total"].sum()


In [ ]:
# Vários cálculos de uma vez, com nomes de colunas claros
resumo_categoria = df_tratado.groupby("Categoria").agg(
    total_vendido=("Valor_Total", "sum"),
    qtd_vendas=("ID_Venda", "count"),
    ticket_medio=("Valor_Total", "mean")
).round(2)

resumo_categoria


## 8. Ordenando e filtrando o resultado


In [ ]:
# Ordenar do maior para o menor valor total
df_tratado.sort_values("Valor_Total", ascending=False)


In [ ]:
# Combinar filtro + ordenação -- só vendas de Eletrônicos, do maior para o menor
(
    df_tratado[df_tratado["Categoria"] == "Eletrônicos"]
    .sort_values("Valor_Total", ascending=False)
)


## 9. Salvando o resultado

Depois de tratar os dados, normalmente você quer salvar o resultado -- seja para reaproveitar depois, seja para entregar para alguém.


In [ ]:
df_tratado.to_excel("vendas_tratado.xlsx", index=False, sheet_name="Vendas_Tratadas")
resumo_categoria.to_excel("resumo_categoria.xlsx", sheet_name="Resumo")

print("Arquivos salvos com sucesso.")


`index=False` evita que o pandas escreva a coluna de índice (0, 1, 2...) como se fosse uma coluna de dados -- normalmente você não quer isso num Excel de entrega.


## 10. Exercícios propostos

Use o `df` original (antes do tratamento) ou o `vendas_exemplo.xlsx` para praticar:

1. Descubra quantas vendas cada `Cliente` fez (dica: `groupby` + `count`).
2. Calcule o valor total vendido por `Produto`, ordenado do maior para o menor.
3. Crie uma coluna `Mes_Venda` extraindo o mês de `Data_Venda` (dica: `.dt.month`).
4. Filtre apenas as vendas com `Valor_Total` acima da média geral.
5. Encontre o produto mais vendido em quantidade (não em valor) por estado.
6. Bônus: tente ler o arquivo `resumo_categoria.xlsx` que você acabou de salvar e conferir se os valores batem com o que você calculou.

Dica geral: sempre que tiver dúvida sobre um método, rode `help(df.metodo)` ou `df.metodo?` num notebook Jupyter para ver a documentação na hora.
